# Qdrant -- Production Vector Database

### High-Performance Search with Advanced Filtering

**Qdrant** is a vector similarity search engine built in Rust for speed and reliability. It supports rich **payload filtering** (structured metadata queries combined with vector search), multiple distance metrics, and horizontal scaling via sharding and replication. Qdrant can run locally in embedded mode (ideal for development) or as a distributed cluster for production workloads. Its filtering capabilities -- including range queries, keyword matches, and nested conditions -- make it especially powerful for RAG pipelines where you need to restrict retrieval to specific document categories, time ranges, or user permissions.

## Installation

The `qdrant-client` Python package provides both a REST/gRPC client for connecting to a remote Qdrant server and an embedded local mode that stores data on disk without needing a separate process. For learning and prototyping, the local-path mode (shown below) is the fastest way to get started.

In [ ]:
# !pip install qdrant-client

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print('✅ Imports successful')

## 1. Initialize Qdrant

Passing a `path` argument to `QdrantClient` creates a local on-disk instance -- no Docker container or remote server required. All data (vectors, payloads, and HNSW indexes) is persisted to the specified directory. For production deployments, you would instead connect to a Qdrant server via `QdrantClient(url="http://host:6333")`.

In [ ]:
client = QdrantClient(path="./qdrant_db")

print("✅ Client created")

## 2. Create Collection

A collection in Qdrant requires a `VectorParams` configuration that specifies the embedding dimensionality (`size`) and the distance metric (`Distance.COSINE`, `Distance.EUCLID`, or `Distance.DOT`). Qdrant builds an HNSW index on the vectors for fast approximate search. Unlike Chroma, Qdrant does not embed text for you -- you must provide pre-computed embedding vectors, giving you full control over which model and preprocessing steps to use.

In [ ]:
from qdrant_client.models import VectorParams, Distance

client.create_collection(
    collection_name="documents",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

print("✅ Collection created")

## 3. Add Vectors

Documents are inserted as `PointStruct` objects, each containing a unique integer or UUID `id`, a `vector` (list of floats), and an optional `payload` dictionary for metadata. The `upsert` method inserts new points or updates existing ones with the same ID, making it safe to re-run. Payloads can store any JSON-serializable data and are indexed automatically for filtering during search.

In [ ]:
import numpy as np

points = [
    PointStruct(
        id=1,
        vector=np.random.random(384).tolist(),
        payload={"text": "ML example"}
    )
]

client.upsert(collection_name="documents", points=points)
print("✅ Vectors added")

## 4. Search

The `search` method takes a `query_vector`, computes approximate nearest neighbors using the HNSW index, and returns the top `limit` results ranked by the collection's distance metric. Each result includes the point's `id`, `score` (similarity), and `payload`. You can optionally pass a `query_filter` parameter to restrict the search to points whose payload fields match specified conditions -- for example, retrieving only documents from a particular category or date range.

In [ ]:
query_vector = np.random.random(384).tolist()

results = client.search(
    collection_name="documents",
    query_vector=query_vector,
    limit=3
)

for hit in results:
    print(f"Score: {hit.score:.4f} - {hit.payload}")